# E1 · The model that finds signal in noise

**Type**: Find the bug · **Level**: ●●○ independent · **Time**: about 30 minutes · **core path ★**

## Objective

A colleague evaluated a model on pure noise data and got a positive cross-validated R². Find
the mistake, fix it with a `Pipeline`, and measure by how much the evaluation was optimistic.

## Dataset

`ds.pure_noise(seed)`: 100 observations, 1000 features and a target, all **independent** Gaussian numbers. There is
no signal to find, so an honest model must have an R² that does not exceed zero.

## Educational purpose

Recognize leakage from feature selection, one of the most common and costly mistakes: the cross-validation
is run correctly, but on data that have already "seen" the target. You will also learn to quantify it.

## Task

1. Run the colleague's cell: `r2_bug` is positive on data that contain nothing. It is not a miracle.
2. Fix `experiment(seed)` so that `r2_ok` is an honest evaluation: the feature selection must happen **inside** every training fold (use a `Pipeline`).
3. Repeat the experiment over 10 seeds (from 0 to 9) in `results` (10 × 2 array, columns `r2_bug`, `r2_ok`) and save in `mean_optimism` the mean of `r2_bug − r2_ok`.
4. Choose in `answer` ("a", "b" or "c") why `r2_ok` comes out **negative** and not simply zero:
   - **a**: there is a mistake in the call to `cross_val_score`;
   - **b**: the model learns random relationships from the training set (10 features chosen among 1000 of noise) and on new data it predicts worse than the plain mean;
   - **c**: `SelectKBest` always makes any model worse.

## Setup

Run this cell without changing it.

In [ ]:
import sys, pathlib
_root = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "common").is_dir())
sys.path.insert(0, str(_root))

import warnings; warnings.filterwarnings("ignore")     # clean output: here the warnings are just noise
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from common import datasets as ds
from common.verify import check, close, done, verify

from sklearn.feature_selection import SelectKBest, f_regression
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import KFold, cross_val_score
from sklearn.pipeline import Pipeline

cv = KFold(5, shuffle=True, random_state=0)

## Your work

### The colleague's code

It runs without errors and prints an encouraging number. Find what is wrong and fix `r2_ok`.

In [ ]:
def experiment(seed):
    """Returns (r2_bug, r2_ok) on a pure noise dataset."""
    X, y = ds.pure_noise(seed=seed)          # X and y are independent: no signal

    # --- the colleague's code ---
    X_sel = SelectKBest(f_regression, k=10).fit_transform(X, y)
    r2_bug = cross_val_score(LinearRegression(), X_sel, y, cv=cv, scoring="r2").mean()

    # --- to fix ---
    r2_ok = r2_bug
    return r2_bug, r2_ok


r2_bug_0, r2_ok_0 = experiment(0)
print(f"colleague's R2: {r2_bug_0:.3f} | fixed R2: {r2_ok_0:.3f}")

In [ ]:
results = np.array([experiment(s) for s in range(10)])
mean_optimism = np.mean(results[:, 0] - results[:, 1])
answer = None

## Check

Run the cell when you think you are done. If a check fails, the message tells you what is wrong. If it fails a second time, a hint appears under the message.

In [ ]:
with verify(hint='`Pipeline([("sel", SelectKBest(f_regression, k=10)), ("lr", LinearRegression())])` passed directly to `cross_val_score(..., X, y, ...)` with the **complete** matrix X: the selection is redone on every training fold.'):
    check(r2_bug_0 > 0.15, "the colleague's number is positive and looks like a good model",
          "r2_bug should be about 0.26 on this dataset: don't modify the colleague's code")
    check(r2_ok_0 < 0, "the fixed R2 is negative: in noise there is nothing to learn",
          "r2_ok must come from a Pipeline with SelectKBest INSIDE the cross-validation")
    check(results.shape == (10, 2) and bool(np.all(results[:, 0] > results[:, 1])),
          "in all 10 seeds the colleague's evaluation is more optimistic",
          "results must have 10 rows and r2_bug > r2_ok in every row")
    check(mean_optimism > 0.4, "the mean optimism is more than 0.4 points of R2",
          "mean_optimism should be the mean of r2_bug - r2_ok over 10 seeds")
    check(answer == "b", "correct explanation", "reread: what does a model learn from 10 random features on 80 observations?")
done()

---
The commented solution is in `solutions/E_find_the_bug/E1_leakage_in_feature_selection.ipynb`. Open it after you pass the check, or after you have really tried.